In [1]:
#lib for handling measurinf time and arrays
import numpy as np
import time

In [2]:
#lib for loading data & prepare it
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

In [3]:
#knn model and metrics to check accuracy
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,classification_report

In [4]:
print("Downloading Fashion MIST dataset(this take time)..")
fashion_mnist=fetch_openml('Fashion-MNIST',version=1, as_frame=False)

In [5]:
x=fashion_mnist.data
y=fashion_mnist.target.astype(int)

print(f"Total dataset size :{x.shape[0]}images,each with {x.shape[1]} pixels.")

Total dataset size :70000images,each with 784 pixels.


In [6]:
#DATA SET SPLIT
x_subset,_,y_subset,_ = train_test_split(
    x,y,
    train_size=12000,
    stratify=y,
    random_state =42
)

In [7]:
#step2 : split that subset into training and testing dataset(10k,2k)
x_train,x_test,y_train,y_test=train_test_split(
    x_subset,y_subset,
    test_size=2000,
    stratify=y_subset,
    random_state=42
)

In [8]:
print(f"Training images :{x_train.shape[0]}")
print(f"Testing images :{x_test.shape[0]}")

Training images :10000
Testing images :2000


In [9]:
#check values before scaling
print(f"Before scaling -> Min:{x_train.min()}, Max:{x_train.max()}")

#rescale to range [0.0, 0.1]
x_train=x_train/255.0
x_test=x_test/255.0

#check values after scaling

print(f"After scaling -> Min:{x_train.min()}, Max: {x_train.max()}")

Before scaling -> Min:0, Max:255
After scaling -> Min:0.0, Max: 1.0


In [15]:
#list of k  values

k_values=[1,3,5,7,9,15]
#dict to store the result
results={}

#print header for the result table
print(f"{'K-values':<8}|{'Accuracy':<10}|{'Prediction Time (seconds)':<25}")
print("-"*50)
for k in k_values:
    #1.initialize the model
    knn=KNeighborsClassifier(n_neighbors= k, metric = 'euclidean',n_jobs=-1)
    #2.train the model(knnjust stores the training data
    knn.fit(x_train,y_train)
    #3.predict labels for test images and measure time
    start_time=time.time()
    y_pred=knn.predict(x_test)
    elapsed_time=time.time()-start_time
    #4.measure accuracy
    acc=accuracy_score(y_test,y_pred)
    #store result
    results[k]={
     "accuracy":acc,
     "time":elapsed_time,
     "prediction":y_pred
    }
    print(f"{k:<8} | {acc*100:<9.2f}% | {elapsed_time:<25.2f}")

K-values|Accuracy  |Prediction Time (seconds)
--------------------------------------------------
1        | 79.30    % | 0.20                     
3        | 80.90    % | 0.20                     
5        | 81.00    % | 0.20                     
7        | 81.10    % | 0.20                     
9        | 80.95    % | 0.20                     
15       | 80.25    % | 0.20                     


In [19]:
# Readable clothing labels mapped to numbers 0-9
class_names=[
    "T-shirt/Top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneakers", "Bag", "Ankle Boot"
]

# Find the K with the highest accuracy
best_k=max(results, key=lambda k: results[k]["accuracy"])

print(f"Best K is: {best_k} with {results[best_k]['accuracy'] * 100:.2f}% accuracy")

# Print classification breakdown
print("Per-Class Classification Report:")
print(classification_report(y_test,results[best_k]["prediction"],target_names=class_names))


Best K is: 7 with 81.10% accuracy
Per-Class Classification Report:
              precision    recall  f1-score   support

 T-shirt/Top       0.73      0.83      0.78       200
     Trouser       0.97      0.94      0.96       200
    Pullover       0.67      0.73      0.70       200
       Dress       0.85      0.83      0.84       200
        Coat       0.74      0.69      0.72       200
      Sandal       1.00      0.76      0.86       200
       Shirt       0.58      0.55      0.56       200
    Sneakers       0.81      0.92      0.86       200
         Bag       0.97      0.94      0.95       200
  Ankle Boot       0.85      0.94      0.89       200

    accuracy                           0.81      2000
   macro avg       0.82      0.81      0.81      2000
weighted avg       0.82      0.81      0.81      2000

